# OP-04 · Référence observée de température (ERA5)

**Notebook opérationnel.**

| | |
|---|---|
| Étape du workflow | E1 — observations de référence (température) |
| Entrées | CDS (T2m horaire ERA5) puis les fichiers journaliers déjà produits |
| Sorties | `DATA_OSF/raw/era5/hourly_0p25/era5_t2m_daily_<année>.nc` · `DATA_OSF/derived/obs/era5/` (moyennes décadaires et mensuelles à 0,25° et 1°, normales 1991–2020, contrôle qualité) |
| Quand l'exécuter | une fois pour constituer l'archive, puis chaque mois pour ajouter le mois écoulé |
| Durée | quelques minutes si seules les dernières années manquent |

**Méthode :** une requête CDS par année (T2m horaire, GRIB), plusieurs années demandées en même temps, puis calcul local de la moyenne, du maximum et du minimum de chaque jour UTC. Le GRIB horaire est supprimé après traitement. Une année déjà traitée est ignorée.

Équivalent en ligne de commande :
```
python scripts/run_download_era5_hourly.py --config config/cycle_YYYYMM.yaml --years 1981 2026 --workers 4
python scripts/run_obs_era5.py --config config/cycle_YYYYMM.yaml
```

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"
ANNEES       = (1981, 2026)   # années à compléter ; celles déjà traitées sont ignorées
WORKERS      = 4              # années demandées simultanément au CDS
REBUILD      = False          # True : recalculer les archives et normales même si la source est inchangée

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
from eccas_s2s.settings import load_cycle
cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")

## 1. Téléchargement et statistiques journalières

In [ ]:
from eccas_s2s.operations import download_era5_hourly
dl = download_era5_hourly.run(CYCLE_CONFIG, ANNEES[0], ANNEES[1], workers=WORKERS)
print(f"\nStatut : {dl.status} — {len(dl.outputs)} année(s) disponible(s)")
for w in dl.warnings:
    print(" ⚠", w)

## 2. Archive dérivée et normales 1991–2020

In [ ]:
from eccas_s2s.operations import obs_era5
ctx = obs_era5.run(CYCLE_CONFIG, rebuild=REBUILD)
print(f"\nStatut : {ctx.status}")
for w in ctx.warnings:
    print(" ⚠", w)

## 3. Contrôle qualité

In [ ]:
paths = obs_era5.derived_paths(cfg)
qc = pd.read_csv(paths["qc"])
print(f"{len(qc)} lignes (mois × variable), {qc.year.min()}–{qc.year.max()}")
display(pd.Series({
    "mois incomplets": int((qc.days_present != qc.days_expected).sum()),
    "valeurs manquantes": int(qc.missing_values.sum()),
    "valeurs hors plage −15…60 °C": int(qc.outside_range.sum()),
    "minimum absolu (°C)": round(float(qc["min"].min()), 1),
    "maximum absolu (°C)": round(float(qc["max"].max()), 1),
}).to_frame("bilan"))
pd.DataFrame([{"fichier": p.name, "taille (Mo)": round(p.stat().st_size / 1e6, 1)}
              for k, p in paths.items() if k != "dir"])

## 4. Contrôle visuel : dernière décade disponible

In [ ]:
from eccas_s2s.viz.maps import map_panel
dek, mon = obs_era5.load_archives(cfg, "0p25")
last = dek.time.values[-1]
fig = map_panel([dek["tmax"].sel(time=last), dek["tmin"].sel(time=last),
                 dek["tmax"].sel(time=last) - dek["tmin"].sel(time=last)],
                ["Tmax", "Tmin", "amplitude diurne"], shapefile=cfg.raw["paths"]["shapefile"],
                cmap="YlOrRd", levels=[0, 5, 10, 15, 20, 25, 30, 35, 40], extend="both",
                cbar_label="°C", suptitle=f"ERA5 — décade du {str(last)[:10]}")

In [ ]:
for c in (dl, ctx):
    print(f"{c.step:24s} {c.status:8s} {c.run_dir / 'manifest.json'}")